# Logging my journey learning LLMs and PyTorch

It is long overdue to start learning PyTorch, and there is no better way to do it than by building a toy LLM. I've gone through a list of papers that helped me to understand how LLMs come to be, and I think it's a good idea to build a toy LLM to better connect the theory with practice.

# Starting the local colab docker image

Use the following command:
```
docker run -it --gpus=all -p 127.0.0.1:9000:8080 --mount type=bind,src="${HOME}/colab/checkpoints",dst=/content/checkpoints us-docker.pkg.dev/colab-images/public/runtime
```

In [1]:
!pwd && ls

/content
checkpoints  sample_data  tiktoken


# Setup PyTorch

In [2]:
import torch

In [3]:
print(f"PyTorch Version: {torch.__version__}")
if (torch.cuda.is_available()):
    device = torch.device("cuda")
    print(f"Using GPU, CUDA version: {torch.version.cuda}, Number of GPUs: {torch.cuda.device_count()}")
else:
    device = torch.device("cpu")

PyTorch Version: 2.10.0+cu128
Using GPU, CUDA version: 12.8, Number of GPUs: 1


# Get a working tokenizer
Let's use OpenAI's tiktoken tokenizer - alghough I am actually interested in building a minimum tokenizer from scratch later.

In [4]:
!git clone https://github.com/openai/tiktoken.git

fatal: destination path 'tiktoken' already exists and is not an empty directory.


In [5]:
import tiktoken

In [6]:
enc = tiktoken.get_encoding("r50k_base")

In [7]:
enc.encode("monkey d luffy")

[49572, 288, 300, 15352]

In [8]:
enc.decode(enc.encode("monkey d luffy"))

'monkey d luffy'

# Get the WebText dataset

In [9]:
from datasets import load_dataset

In [10]:
openwebtext = load_dataset("openwebtext")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/80 [00:00<?, ?it/s]

In [11]:
openwebtext

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 8013769
    })
})

In [12]:
openwebtext['train']

Dataset({
    features: ['text'],
    num_rows: 8013769
})

## Test tokenizing the training data

In [13]:
enc.encode(openwebtext['train'][42]['text'])

[5159,
 8305,
 370,
 868,
 510,
 1165,
 1903,
 290,
 1719,
 2761,
 25446,
 736,
 284,
 3993,
 743,
 423,
 257,
 4633,
 2928,
 319,
 262,
 2612,
 11,
 257,
 2050,
 2523,
 198,
 198,
 8061,
 508,
 423,
 5876,
 38193,
 572,
 284,
 3993,
 743,
 307,
 379,
 3220,
 2526,
 286,
 2612,
 5287,
 11,
 4837,
 910,
 13,
 198,
 198,
 464,
 2050,
 11,
 3199,
 287,
 262,
 3427,
 8894,
 4913,
 11,
 3940,
 517,
 621,
 2026,
 11,
 830,
 661,
 329,
 1367,
 812,
 13,
 198,
 198,
 29193,
 1043,
 883,
 508,
 6989,
 1811,
 12513,
 286,
 3595,
 3993,
 547,
 517,
 1884,
 284,
 1205,
 262,
 4006,
 11,
 287,
 543,
 262,
 2612,
 10143,
 284,
 8901,
 6105,
 13,
 198,
 198,
 38897,
 910,
 2252,
 2267,
 318,
 2622,
 284,
 766,
 611,
 257,
 3092,
 286,
 3993,
 5640,
 2612,
 5287,
 393,
 262,
 2792,
 318,
 517,
 3716,
 13,
 198,
 198,
 1,
 42332,
 867,
 286,
 262,
 1243,
 326,
 4646,
 262,
 2863,
 286,
 2612,
 5287,
 635,
 4646,
 47104,
 26,
 922,
 5496,
 11,
 5517,
 11,
 3463,
 2994,
 290,
 407,
 9216,
 1583,
 5045,
 

# Embedding layer

In [14]:
vocabulary_size = enc.n_vocab
d_model = 768

embedding_layer = torch.nn.Embedding(vocabulary_size, d_model)

Here we encode the phrase "monkey d luffy", which is then fed into the embedding_layer,
which produces d_model=768 embeddings, one 768 dimentional vector per token

In [15]:
embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")]))

tensor([[[ 6.9435e-01, -9.3444e-01,  3.3931e-01,  ...,  1.1523e+00,
          -1.6283e+00, -1.7293e+00],
         [ 3.4150e-01,  2.8299e-01,  5.7936e-01,  ...,  9.1126e-01,
          -9.4286e-01, -1.1795e+00],
         [ 5.0447e-01,  1.7672e+00,  2.1530e+00,  ..., -1.1830e+00,
          -8.0873e-01,  1.2038e+00],
         [-1.1450e+00,  1.8673e-03,  1.0960e+00,  ...,  4.2742e-01,
          -2.2140e-01,  4.1243e-01]]], grad_fn=<EmbeddingBackward0>)

In [16]:
embedding_layer.weight.shape

torch.Size([50257, 768])

Here we can see that the embedding layer is rather simple - it is a n_vocab x d_model matrix. So the embedding process is simply taking w[encoding].

## Positional Encoding

In [17]:
class LearnablePositionalEncoding(torch.nn.Module):
    def __init__(self, max_seq_len, d_model):
        super().__init__()
        self.pos_embeddings = torch.nn.Embedding(max_seq_len, d_model)
    def forward(self, x):
        # x is of shape (batch_size, seq_len, d_model)
        seq_len = x.size(1)

        position_ids = torch.arange(seq_len, dtype=torch.long, device=x.device)

        pos_enc = self.pos_embeddings(position_ids)
        return x + pos_enc


In [18]:
pos_embed_layer = LearnablePositionalEncoding(1024, d_model)

In [19]:
pos_embed_layer(embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")])))

tensor([[[-0.2833, -2.0413, -0.4728,  ...,  0.1253, -0.6488, -1.7745],
         [ 0.7538,  0.5073,  1.7216,  ...,  0.7838, -1.1186, -2.1892],
         [ 1.3060,  2.2537,  3.9647,  ..., -1.1759, -0.8046,  3.1292],
         [-0.7770,  0.2044,  3.0710,  ..., -0.4874, -0.7231,  0.7237]]],
       grad_fn=<AddBackward0>)

# Decoder Only Transformer Layer

This is the key part of the transformer architecture. Each attention layer is made of multi-head attention, normalization, and position-wise feedforward.

In GPT-2, normalization is moved at the beginning of each layer.

In [20]:
embedding = embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")]))

## Layer Normalization

https://arxiv.org/pdf/1607.06450

In [21]:
layer_norm = torch.nn.LayerNorm(d_model)

In [22]:
x = layer_norm(embedding)

In [23]:
x

tensor([[[ 0.7690, -0.9682,  0.3903,  ...,  1.2574, -1.7083, -1.8160],
         [ 0.3605,  0.3030,  0.5943,  ...,  0.9206, -0.9021, -1.1347],
         [ 0.4781,  1.7093,  2.0854,  ..., -1.1672, -0.8023,  1.1600],
         [-1.1822, -0.0362,  1.0571,  ...,  0.3890, -0.2593,  0.3740]]],
       grad_fn=<NativeLayerNormBackward0>)

In [24]:
x.shape

torch.Size([1, 4, 768])

## Multi-Head Attention

In [25]:
n_head = 12

# project into Q, K, V
projection = torch.nn.Linear(d_model, d_model * 3)

In [26]:
projection(x).shape

torch.Size([1, 4, 2304])

In [27]:
proj = projection(x).view(1, -1, 12, d_model // n_head * 3).permute(0, 2, 1, 3)

In [28]:
q, k, v = proj.chunk(3, dim=3)

In [29]:
import math
atten = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_model)

In [30]:
mask = torch.tril(torch.ones_like(atten[0][0]))

In [31]:
atten = atten.masked_fill(mask == 0, -float('inf'))

In [32]:
atten

tensor([[[[ 1.1003e-01,        -inf,        -inf,        -inf],
          [ 1.6854e-01, -3.0857e-03,        -inf,        -inf],
          [ 4.7765e-02,  4.0595e-02,  5.2447e-02,        -inf],
          [ 6.8058e-03, -5.4817e-02, -4.5665e-02,  4.0810e-02]],

         [[ 1.6380e-01,        -inf,        -inf,        -inf],
          [ 1.3455e-01, -5.9358e-02,        -inf,        -inf],
          [-4.7175e-02, -3.1813e-02, -9.6447e-02,        -inf],
          [ 3.4966e-02, -4.0362e-02, -2.5441e-01, -1.3530e-01]],

         [[-3.3066e-02,        -inf,        -inf,        -inf],
          [-7.0966e-02,  1.0879e-01,        -inf,        -inf],
          [-2.1374e-02, -1.1714e-02, -2.9506e-02,        -inf],
          [-6.4836e-02,  4.9714e-02,  6.4396e-02, -6.1821e-02]],

         [[ 4.9098e-02,        -inf,        -inf,        -inf],
          [ 4.3697e-02,  1.3762e-04,        -inf,        -inf],
          [ 1.0439e-01, -2.3026e-02, -1.8080e-01,        -inf],
          [-1.1595e-01,  9.2185e-0

In [33]:
softmax = torch.nn.Softmax(dim=-1)

atten = softmax(atten)

In [34]:
atten

tensor([[[[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5428, 0.4572, 0.0000, 0.0000],
          [0.3336, 0.3312, 0.3352, 0.0000],
          [0.2549, 0.2396, 0.2418, 0.2637]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5483, 0.4517, 0.0000, 0.0000],
          [0.3370, 0.3422, 0.3208, 0.0000],
          [0.2841, 0.2635, 0.2127, 0.2396]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4552, 0.5448, 0.0000, 0.0000],
          [0.3332, 0.3364, 0.3305, 0.0000],
          [0.2346, 0.2631, 0.2670, 0.2353]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5109, 0.4891, 0.0000, 0.0000],
          [0.3799, 0.3345, 0.2856, 0.0000],
          [0.2256, 0.2556, 0.2749, 0.2439]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4563, 0.5437, 0.0000, 0.0000],
          [0.3521, 0.3267, 0.3212, 0.0000],
          [0.2519, 0.2676, 0.2317, 0.2488]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5368, 0.4632, 0.0000, 0.0000],
          [0.3410, 0.3

In [35]:
torch.matmul(atten, v).transpose(1, 2).reshape(1, 4, d_model)

tensor([[[ 0.1286, -0.2221, -0.0141,  ..., -0.5507, -0.2520, -0.4266],
         [ 0.1458, -0.0121,  0.3970,  ...,  0.1452,  0.3857, -0.3909],
         [ 0.3657, -0.0798,  0.5633,  ...,  0.4123,  0.0228, -0.2665],
         [ 0.3360, -0.0889,  0.2182,  ...,  0.4576, -0.0537, -0.0390]]],
       grad_fn=<UnsafeViewBackward0>)

### Putting it together

In [36]:
import torch.nn.functional as F
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # project into K Q V
        self.input_linear = torch.nn.Linear(d_model, d_model * 3)
    def forward(self, x):
        batch_size, sequence_length, _ = x.size()
        proj = self.input_linear(x).view(batch_size, sequence_length, self.n_heads, self.d_head * 3).permute(0, 2, 1, 3)
        q, k, v = proj.chunk(3, dim=-1)

        # Standard Transformer scaling: 1/sqrt(d_head)
        atten = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        mask = torch.tril(torch.ones_like(atten[0][0]))
        atten = atten.masked_fill(mask == 0, -float('inf'))
        atten = F.softmax(atten, dim=-1)

        return torch.matmul(atten, v).transpose(1, 2).reshape(batch_size, sequence_length, self.d_model)

In [37]:
multi_head_attention = MultiHeadAttention(d_model, n_head)
multi_head_attention(x)

tensor([[[ 3.0373e-01,  2.8010e-01, -3.6256e-01,  ..., -2.5042e-01,
          -1.0357e-04, -8.4681e-01],
         [-1.6724e-01,  4.5401e-01, -1.8507e-02,  ..., -2.2757e-01,
          -7.0133e-02, -3.5559e-01],
         [-1.8175e-01,  4.0615e-01, -1.2067e-01,  ..., -4.8436e-01,
          -2.4609e-02, -5.1288e-01],
         [-1.8244e-01,  1.6640e-01, -2.3352e-01,  ..., -2.7828e-01,
           1.9654e-01, -3.6578e-01]]], grad_fn=<UnsafeViewBackward0>)

## Position-wise Feed Forward

In [38]:
d_hidden = 3072

In [39]:
class DenseFeedForward(torch.nn.Module):
    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.model = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(d_hidden, d_model)
        )
    def forward(self, x):
        return self.model(x)

In [40]:
y = multi_head_attention(x)

In [41]:
dff = DenseFeedForward(d_model, d_hidden)

In [42]:
dff(y)

tensor([[[-0.0295, -0.1212, -0.0569,  ..., -0.1289, -0.0648, -0.1917],
         [-0.1741, -0.1494,  0.0010,  ..., -0.1255, -0.0289, -0.2101],
         [-0.0017, -0.1571,  0.0447,  ..., -0.0756, -0.0527, -0.1592],
         [ 0.0303, -0.1816,  0.0224,  ..., -0.0283, -0.0368, -0.1665]]],
       grad_fn=<ViewBackward0>)

## The Full Transformer Layer

In [43]:
class TransformerDecoderOnly(torch.nn.Module):
    def __init__(self, d_model, d_hidden, n_head):
        super().__init__()
        self.layer_norm_0 = torch.nn.LayerNorm(d_model)
        self.multi_head_attention = MultiHeadAttention(d_model, n_head)
        self.layer_norm_1 = torch.nn.LayerNorm(d_model)
        self.dff = DenseFeedForward(d_model, d_hidden)
    def forward(self, x):
        # layer normalization first
        x = self.layer_norm_0(x)

        # multi-head attention and residual
        identity = x
        x = self.multi_head_attention(x)
        x = x + identity

        # layer normalization before dense feed forward
        x = self.layer_norm_1(x)

        # dense feed forward and residual
        identity = x
        x = self.dff(x)
        return x + identity

In [44]:
transformer_decoder_only = TransformerDecoderOnly(d_model, d_hidden, n_head)

transformer_decoder_only(embedding)

tensor([[[ 0.3577, -0.3653, -0.1695,  ...,  2.0745, -1.8934, -1.3332],
         [ 0.2630,  0.2398,  0.4712,  ...,  1.4449, -1.3981, -1.1250],
         [ 0.2737,  1.1934,  1.7522,  ..., -0.5232, -1.1067,  0.8708],
         [-0.9188,  0.0954,  0.9522,  ...,  1.1281, -0.5194,  0.4071]]],
       grad_fn=<AddBackward0>)

# Output Layer

The output layer is rather simple. matrix multiply with the input embedding matrix, and soft max. We can then reverse the tokenization.

In [45]:
torch.matmul(transformer_decoder_only(embedding), embedding_layer.weight.transpose(0, 1))

tensor([[[-38.6808,  14.2083,  11.6528,  ...,  16.8211,  -9.6074, -31.0878],
         [  4.1043,   6.8721,   8.0666,  ...,   0.4621,   2.8368, -44.3870],
         [-47.2303,  -9.7047,   4.6238,  ..., -34.5303,   1.1735, -13.6582],
         [  4.4154, -32.4294,  25.6011,  ..., -72.1586,  -7.1835,  15.2255]]],
       grad_fn=<UnsafeViewBackward0>)

In [46]:
class Output(torch.nn.Module):
    def __init__(self, embedding_layer_weights):
        super().__init__()
        # Removed the extra LayerNorm here as it can skew initial logits
        self.embedding_layer_weights = embedding_layer_weights

    def forward(self, x):
        # x: (batch, seq, d_model)
        # weights: (vocab, d_model)
        return torch.matmul(x, self.embedding_layer_weights.transpose(0, 1))

# The full model

In [60]:
class ToyGPT(torch.nn.Module):
    def __init__(self, d_model, d_hidden, n_head, vocab_size, layers, max_seq_len):
        super().__init__()
        self.embedding_layer = torch.nn.Embedding(vocab_size, d_model)
        self.pos_encoding_layer = LearnablePositionalEncoding(max_seq_len, d_model)
        self.transformers = torch.nn.Sequential(*[TransformerDecoderOnly(d_model, d_hidden, n_head) for _ in range(layers)])
        # Standard GPT-2: Final LayerNorm before the output head
        self.ln_f = torch.nn.LayerNorm(d_model)
        self.output_layer = Output(self.embedding_layer.weight)

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, torch.nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, torch.nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):
        x = self.embedding_layer(x)
        x = self.pos_encoding_layer(x)
        x = self.transformers(x)
        x = self.ln_f(x) # Apply the missing final normalization
        return self.output_layer(x)

# Training

In [48]:
batch_size = 48
max_seq_len = 1024

In [49]:
enc._special_tokens

{'<|endoftext|>': 50256}

Pre-tokenize the training data to avoid CPU bottleneck.

In [50]:
from datasets import Dataset

rows = openwebtext["train"].num_rows
bos_token = '<|endoftext|>'

def tokenize(examples):
    out = enc.encode(examples['text']+ bos_token, allowed_special={bos_token})
    return {'tokenized_text': out}

tokenized_data = openwebtext["train"].map(tokenize, remove_columns=openwebtext["train"].column_names, num_proc=32)


In [51]:
def do_checkpoint(model, optimizer, running_encoding, row_number):
    torch.save({
        'row_number': row_number,
        'running_encoding': running_encoding,
        'optimizer_state_dict': optimizer.state_dict(),
        'model_state_dict': model.state_dict(),
    }, 'checkpoints/check_point_{}'.format(row_number))

In [65]:
import gc
import math
from tqdm.notebook import tqdm

def get_lr(step, warmup_steps, max_steps, max_lr, min_lr):
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    if step > max_steps:
        return min_lr
    decay_ratio = (step - warmup_steps) / (max_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)

@torch.no_grad()
def generate_sample(model, prompt="The secret to life is", max_new_tokens=50):
    model.eval()
    input_ids = torch.tensor([enc.encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        cond_ids = input_ids[:, -max_seq_len:]
        logits = model(cond_ids)
        logits = logits[:, -1, :]
        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        input_ids = torch.cat((input_ids, next_token), dim=1)
        if next_token.item() == enc._special_tokens[bos_token]:
            break
    out_text = enc.decode(input_ids[0].tolist())
    model.train()
    return out_text

def train(start_row_number = 0, check_point_interval = 100000, summary_writer = None):
    tokens_per_batch = (batch_size + 1) * max_seq_len // 2
    max_lr = 0.0006
    min_lr = max_lr * 0.1
    warmup_steps = 20000
    max_steps = rows

    encoding = []
    model = ToyGPT(d_model, d_hidden, n_head, enc.n_vocab, 12, max_seq_len)
    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.1)
    model.to(device)
    model.train()
    model = torch.compile(model)

    if start_row_number != 0:
        checkpoint = torch.load('checkpoints/check_point_{}'.format(start_row_number))
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        encoding = checkpoint['running_encoding']

    write_summary_threshold = start_row_number
    check_point_threshold = (start_row_number + check_point_interval) // check_point_interval * check_point_interval
    data = tokenized_data.select(range(start_row_number, tokenized_data.num_rows)).to_iterable_dataset(num_shards=8)

    for row in enumerate(tqdm(data, total=rows, initial=start_row_number)):
        cur_row_number = start_row_number + row[0]
        lr = get_lr(cur_row_number, warmup_steps, max_steps, max_lr, min_lr)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        encoding.extend([enc._special_tokens[bos_token]] + row[1]['tokenized_text'])

        if len(encoding) > tokens_per_batch:
            residual = encoding[tokens_per_batch:]
            encoding = encoding[0:tokens_per_batch + 1]
            batch = torch.empty((batch_size, max_seq_len), dtype=torch.long, device=device)
            target = torch.empty((batch_size, max_seq_len), dtype=torch.long, device=device)

            step_size = max_seq_len // 2
            for i in range(0, batch_size):
                start_idx = i * step_size
                batch[i] = torch.LongTensor(encoding[start_idx : start_idx + max_seq_len])
                target[i] = torch.LongTensor(encoding[start_idx + 1 : start_idx + max_seq_len + 1])

            optimizer.zero_grad()
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                out = model(batch)
                loss = F.cross_entropy(out.view(-1, enc.n_vocab), target.view(-1))

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            encoding = residual

            if summary_writer is not None and cur_row_number >= write_summary_threshold:
                write_summary_threshold = (write_summary_threshold + 1000) // 1000 * 1000
                summary_writer.add_scalar('Loss/train', loss.item(), cur_row_number)
                summary_writer.add_scalar('LR/train', lr, cur_row_number)
                summary_writer.flush()

            if (cur_row_number + 1 >= check_point_threshold):
                print(f"\nCheckpointing at {cur_row_number + 1}. Loss: {loss.item():.4f}")
                do_checkpoint(model, optimizer, encoding, cur_row_number + 1)

                # Generate and log sample
                sample = generate_sample(model)
                print(f"Sample Generation: {sample}\n")
                if summary_writer is not None:
                    summary_writer.add_text('Generation/sample', sample, cur_row_number + 1)
                    summary_writer.flush()

                check_point_threshold = (cur_row_number + 1 + check_point_interval) // check_point_interval * check_point_interval

            gc.collect()

### Validation: Architecture and Initial Loss
For a randomly initialized model, the expected cross-entropy loss is $-\ln(1/V) = \ln(V)$, where $V$ is the vocabulary size. Let's verify this and check the output shapes.

In [53]:
import math

# 1. Check Output Shapes
test_model = ToyGPT(d_model, d_hidden, n_head, vocabulary_size, layers=1, max_seq_len=max_seq_len)
test_input = torch.randint(0, vocabulary_size, (1, 8)) # Batch size 1, Seq len 8
with torch.no_grad():
    logits = test_model(test_input)

print(f"Input shape: {test_input.shape}")
print(f"Output (logits) shape: {logits.shape} (Expected: [1, 8, {vocabulary_size}])")

# 2. Check Initial Loss
targets = torch.randint(0, vocabulary_size, (1, 8))
loss = F.cross_entropy(logits.view(-1, vocabulary_size), targets.view(-1))

theoretical_loss = math.log(vocabulary_size)
print(f"\nTheoretical Initial Loss: {theoretical_loss:.4f}")
print(f"Actual Initial Loss:      {loss.item():.4f}")
print(f"Difference:              {abs(loss.item() - theoretical_loss):.4f}")

if abs(loss.item() - theoretical_loss) < 0.5:
    print("\n✅ The initial loss is consistent with random initialization.")
else:
    print("\n⚠️ Initial loss is significantly different from theory. Check initialization or scaling.")

Input shape: torch.Size([1, 8])
Output (logits) shape: torch.Size([1, 8, 50257]) (Expected: [1, 8, 50257])

Theoretical Initial Loss: 10.8249
Actual Initial Loss:      11.5472
Difference:              0.7223

⚠️ Initial loss is significantly different from theory. Check initialization or scaling.


In [54]:
%load_ext tensorboard

In [55]:
from torch.utils.tensorboard import SummaryWriter

summary_writer = SummaryWriter('./checkpoints/adamw_run_lr_0006')

In [67]:
%tensorboard --logdir checkpoints/adamw_run_lr_0006 --bind_all


<IPython.core.display.Javascript object>

In [ ]:
train(start_row_number=200004, check_point_interval=100000, summary_writer = summary_writer)

  2%|2         | 200004/8013769 [00:00<?, ?it/s]